In [ ]:
from kiteconnect import KiteConnect
import pandas as pd
import datetime as dt
import os

In [ ]:
request_token=open("request_token.txt","r").read()
key_secrets=open("api_key.txt","r").read().split()
kite = KiteConnect(api_key=key_secrets[0])
data = kite.generate_session(request_token=request_token,api_secret=key_secrets[1])

In [ ]:
# Get dump of all NSE instruments
instruments_dump=kite.instruments("NSE")
instruments_df=pd.DataFrame(instruments_dump)
instruments_df.to_csv("NSE_Instruments.csv",index=False)

In [ ]:
def instrumentLookup(instruments_df,symbol):
    """Looks up instrument token for a given script from instrument dump"""
    try:
        return instruments_df[instruments_df.tradingsymbol==symbol].instrument_token.values[0]
    except:
        return -1


def fetchOHLC(ticker,interval,duration):
    """extracts historical data and outputs in the form of dataframe"""
    instrument = instrumentLookup(instruments_df,ticker)
    data = pd.DataFrame(kite.historical_data(instrument,dt.date.today()-dt.timedelta(duration), dt.date.today(),interval))
    data.set_index("date",inplace=True)
    return data

In [ ]:

def rsi(df, n):
    """Function to calculate the Relative Strength Index (RSI).
    
    Parameters:
    df (pd.DataFrame): DataFrame containing closing prices with a 'close' column.
    n (int): The period over which to calculate the RSI.
    
    Returns:
    pd.Series: The RSI values.
    """
    delta = df["close"].diff().dropna()
    u = delta * 0
    d = u.copy()
    u[delta > 0] = delta[delta > 0]
    d[delta < 0] = -delta[delta < 0]
    u[u.index[n-1]] = np.mean( u[:n]) # first value is average of gains
    u = u.drop(u.index[:(n-1)])
    d[d.index[n-1]] = np.mean( d[:n]) # first value is average of losses
    d = d.drop(d.index[:(n-1)])
    rs = u.ewm(com=n,min_periods=n).mean()/d.ewm(com=n,min_periods=n).mean()
    return 100 - 100 / (1+rs)



In [ ]:
ohlc = fetchOHLC("INFY","5minute",5)
rsi = rsi(ohlc,14)